# Component 10: Constrained Report Generator (BioGPT-Large)

This notebook demonstrates the setup for the final component: generating a clinical report from the structured JSON using BioGPT. We also verify the model freezing constraints and the faithfulness checker.

In [ ]:
import sys
from pathlib import Path
import json
import torch

# Ensure our src imports work
sys.path.append('..')

from src.components.component10_biogpt import Component10ReportGenerator, FaithfulnessChecker

## 1. Init Model and Verify Frozen Layers
BioGPT-Large (1.5B parameters). Layers 1-10 are frozen. Layers 11-12 and the LM head are trainable for optional fine-tuning.

In [ ]:
# Instantiate the model (using mock to avoid a 6GB download in the notebook, 
# but the code behaves identically for the real huggingface model)
generator = Component10ReportGenerator(use_mock=True)
generator.eval()

# Verify parameter counts and frozen layers
frozen_params = sum(p.numel() for p in generator.model.biogpt.layers[:10].parameters() if not p.requires_grad)
trainable_params = sum(p.numel() for p in generator.model.biogpt.layers[10:].parameters() if p.requires_grad)
lm_head_trainable = sum(p.numel() for p in generator.model.output_projection.parameters() if p.requires_grad)

print("BioGPT-Large Freezing Verification:")
print(f"Frozen Parameters (Layers 1-10): {frozen_params:,}")
print(f"Trainable Parameters (Layers 11-12): {trainable_params:,}")
print(f"Trainable LM Head: {lm_head_trainable:,}")

## 2. Few-Shot Prompt Generation
We take the output from Component 9 (JSON) and format it into a structural prompt with 3 revised few-shot examples.

In [ ]:
# This simulates the JSON passed from Component 9
evidence_json = {
    "patient_id": "TBX_8014",
    "segmentation": {
        "n_distinct_lesions": 2, 
        "lesion_area_cm2": 15.0
    },
    "scoring": {
        "ALP": 48.5, 
        "cavity_flag": 1, 
        "severity": "severe",
        "cavitation_confidence": "radiographic-only"
    },
    "pathology_flags": {
        "consolidation": True,
        "cavitation": True
    }
}

prompt = generator.format_prompt(evidence_json)
print("=== GENERATED PROMPT INTENDED FOR BIOGPT ===\n")
print(prompt)
print("\n===========================================")

## 3. Faithfulness Checker
Verifies the produced text strictly follows the JSON without hallucinating severity or non-existent features.

In [ ]:
checker = FaithfulnessChecker()

# Imagine BioGPT generated this faithful report:
faithful_report = "Two distinct lesions are observed covering approximately 15.0 cm2. The affected lung percentage is 48.5%, indicative of a severe disease pattern. Radiographic evidence points toward consolidation and cavitation."

# Imagine BioGPT hallucinated milder severity or missed the cavity:
hallucinated_report = "A single lesion is seen. The affected lung percentage is indicative of a mild disease pattern. No cavitation is found."

print(f"Faithful Report Passed? : {checker.verify_report(faithful_report, evidence_json)}")
print(f"Hallucinated Report Passed? : {checker.verify_report(hallucinated_report, evidence_json)}")